# 1b — Install LIBERO-Plus assets

Run this notebook after `01_setup_and_preflight.ipynb` and before `02_freeze_design_and_import.ipynb`. It downloads the pinned LIBERO-Plus asset archive, extracts it, selects the real top-level asset directory, and installs it under the pinned clone. The operation is idempotent: rerunning it leaves an existing nonempty installation untouched.

The download is approximately 6.4 GB and extraction temporarily requires substantially more free disk space. Do not interrupt the extraction.

In [ ]:
import shutil
import subprocess
from pathlib import Path

home = Path.home()
ood_python = home / "venv-stage1-ood/bin/python"
libero_plus = home / "LIBERO-plus"
target = libero_plus / "libero/libero/assets"
download_dir = home / "libero-plus-download"
extract_dir = home / "libero-plus-assets-extracted"

for required in (ood_python, libero_plus):
    if not required.exists():
        raise SystemExit(f"STOP: missing {required}; finish notebook 1 first")

usage = shutil.disk_usage(home)
print(f"Free disk: {usage.free / 1024**3:.1f} GiB")
if usage.free < 15 * 1024**3 and not (target.is_dir() and any(target.iterdir())):
    raise SystemExit("STOP: less than 15 GiB free; asset download/extraction may fill the disk")

print("Target:", target)


In [ ]:
if target.is_dir() and any(target.iterdir()):
    print("Assets already installed; download skipped:", target)
else:
    download_dir.mkdir(exist_ok=True)
    extract_dir.mkdir(exist_ok=True)

    download_code = r'''
from huggingface_hub import hf_hub_download
from pathlib import Path
path = hf_hub_download(
    repo_id="Sylvest/LIBERO-plus",
    repo_type="dataset",
    filename="assets.zip",
    local_dir=str(Path.home() / "libero-plus-download"),
)
print(path)
'''
    subprocess.run([str(ood_python), "-c", download_code], check=True)

    archive = download_dir / "assets.zip"
    if not archive.is_file():
        raise RuntimeError(f"download did not create {archive}")
    print(f"Archive: {archive} ({archive.stat().st_size / 1024**3:.2f} GiB)")

    marker = extract_dir / ".extraction_complete"
    if not marker.exists():
        print("Extracting; do not interrupt...")
        subprocess.run(["unzip", "-q", str(archive), "-d", str(extract_dir)], check=True)
        marker.write_text("complete\n")
    else:
        print("Completed extraction already present; extraction skipped")

    candidates = [path for path in extract_dir.rglob("assets") if path.is_dir()]
    if not candidates:
        raise RuntimeError("no assets directory found in extracted archive")

    def directory_size(path):
        return sum(item.stat().st_size for item in path.rglob("*") if item.is_file())

    sizes = [(directory_size(path), path) for path in candidates]
    for size, path in sorted(sizes, reverse=True):
        print(f"candidate {size / 1024**3:.2f} GiB: {path}")
    source = max(sizes)[1]

    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        raise RuntimeError(f"refusing to overwrite unexpected existing target: {target}")
    shutil.move(str(source), str(target))
    print("Installed assets at:", target)


In [ ]:
assert target.is_dir(), target
entries = list(target.iterdir())
assert entries, f"empty asset directory: {target}"
installed_bytes = sum(item.stat().st_size for item in target.rglob("*") if item.is_file())
print("PASS: LIBERO-Plus assets ready")
print("path:", target)
print("top-level entries:", len(entries))
print(f"installed size: {installed_bytes / 1024**3:.2f} GiB")
print("You may now rerun notebook 02_freeze_design_and_import.ipynb")
